# Ollama with Python

- [ollama.com](https://ollama.com/)
- [ollama.com/models](https://ollama.com/models)

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 8.95 ms, sys: 10.9 ms, total: 19.8 ms
Wall time: 1.22 s


In [2]:
%pip install -qU ollama asyncio ipython-autotime --prefer-binary

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 192 μs (started: 2025-06-18 13:52:53 -07:00)


In [3]:
%pip list --format=columns | grep ollama

ollama                                   0.5.1
Note: you may need to restart the kernel to use updated packages.
time: 708 ms (started: 2025-06-18 13:52:53 -07:00)


## Define varibles

In [4]:
my_model='llama3.2'
my_sentence='Why is the sky blue?'

print(f"Model: {my_model} ; sentence: {my_sentence} ")

Model: llama3.2 ; sentence: Why is the sky blue? 
time: 416 μs (started: 2025-06-18 13:52:53 -07:00)


## simple usage

In [5]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': my_sentence,
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with the tiny molecules of gases in the Earth's atmosphere.

When sunlight enters the Earth's atmosphere, it encounters tiny molecules of nitrogen and oxygen. These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described the phenomenon in the late 19th century.

As a result of this scattering, the blue light is dispersed throughout the atmosphere, giving the sky its blue color. The amount of scattering that occurs depends on the wavelength of the light and the size of the molecules. Since blue light has a shorter wavelength than red light, it scatters more easily, which is why the sky appears blue.

Here's a rough breakdown of what happens:

* Blue light (shorter wavelength) is scattered by the tiny molecules in

## Streaming responses

In [6]:
from ollama import chat

stream = chat(
    model=my_model,
    messages=[{'role': 'user', 'content': my_sentence}],
    stream=True,
)

for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)


The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens:

1. **Sunlight enters Earth's atmosphere**: When sunlight enters our atmosphere, it consists of a broad spectrum of colors, including all the colors of the visible spectrum (red, orange, yellow, green, blue, indigo, and violet).
2. **Light interacts with tiny molecules**: As sunlight travels through the atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2). These molecules are much smaller than the wavelength of light.
3. **Scattering occurs**: When light hits a molecule, it scatters in all directions. This scattering effect is more pronounced for shorter wavelengths (like blue and violet) because these wavelengths are more easily deflected by the small molecules.
4. **Blue light is scattered more**: As a result of this scattering, the blue light is distributed t

## Custom client

In [7]:
from ollama import Client

client = Client(
  host='http://localhost:11434',
  headers={'Content-Type': 'application/json'}
)
response = client.chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': my_sentence,
  },
])

response

ChatResponse(model='llama3.2', created_at='2025-06-18T20:53:13.692183Z', done=True, done_reason='stop', total_duration=5183033458, load_duration=12195791, prompt_eval_count=31, prompt_eval_duration=20050500, eval_count=267, eval_duration=5150425334, message=Message(role='assistant', content="The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with tiny molecules of gases in the Earth's atmosphere.\n\nHere's what happens:\n\n1. Sunlight enters the Earth's atmosphere and consists of a spectrum of colors, including all the colors of the visible light.\n2. When sunlight hits the tiny molecules of gases such as nitrogen (N2) and oxygen (O2) in the atmosphere, it scatters in all directions.\n3. The shorter (blue) wavelengths of light are scattered more than the longer (red) wavelengths because they are more easily deflected by the smaller molecules.\n4. As a result, the blue light is dispersed throughout the atmosphere, giving the sky its blue

time: 5.2 s (started: 2025-06-18 13:53:08 -07:00)


## Generate embeddings with a model

In [8]:
from ollama import embed

response = embed(model= my_model, input=my_sentence)
print(response['embeddings'])

[[-0.014822339, 0.006883563, -0.024879357, -0.010968463, 0.008566333, 0.013449837, 0.016976569, -0.006796976, 2.1612457e-05, -0.014977558, -0.012403325, 0.013154302, 0.01109737, 0.02421846, -0.028001877, 0.024571266, 0.0099327, 0.006868269, 0.009581534, -0.014612331, -0.0030204076, -0.00065914314, -0.00026611995, 0.020402247, 0.01757082, 0.008109578, -0.0071796402, -0.009318115, 0.03736044, 0.00041696525, -0.016444571, -0.0056094434, 0.00305329, 0.020939514, 0.01885041, -0.013584638, 0.011069951, 0.009739772, -0.0071502854, -0.003120606, -0.014066324, 0.0028459544, 0.010807796, 0.0035745203, -0.009187045, -0.018975912, 0.024012389, 0.009891413, -0.0043805437, 0.0027884196, 0.011050143, 0.010873832, 0.006048481, 0.025744557, 0.007869001, 0.011359033, 0.021624513, -0.0377999, 0.027300961, 0.012038383, 0.016865093, 0.00015775049, 0.010800233, -0.022625964, 0.04749556, -0.007371454, 0.01597823, -0.03001254, -0.0056250207, -0.006960269, -0.010920165, 0.015077568, 0.0031494466, 0.0014062662,

## Show model status with CPU/GPU usage

In [9]:
from ollama import ProcessResponse, chat, ps, pull

# Ensure at least one model is loaded
response = pull(my_model, stream=True)
progress_states = set()
for progress in response:
  if progress.get('status') in progress_states:
    continue
  progress_states.add(progress.get('status'))
  print(progress.get('status'))

print('\n')

print('Waiting for model to load... \n')
chat(model='llama3.2', messages=[{'role': 'user', 'content': my_sentence}])


response: ProcessResponse = ps()
for model in response.models:
  print('Model: ', model.model)
  print('  Digest: ', model.digest)
  print('  Expires at: ', model.expires_at)
  print('  Size: ', model.size)
  print('  Size vram: ', model.size_vram)
  print('  Details: ', model.details)
  print('\n')

pulling manifest
pulling dde5aa3fc5ff
pulling 966de95ca8a6
pulling fcc5a6bec9da
pulling a70ff7e570d9
pulling 56bb8bd477a5
pulling 34bb5ab01051
verifying sha256 digest
writing manifest
success


Waiting for model to load... 

Model:  llama3.2:latest
  Digest:  a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72
  Expires at:  2025-06-18 13:58:23.449400-07:00
  Size:  4030033920
  Size vram:  4030033920
  Details:  parent_model='' format='gguf' family='llama' families=['llama'] parameter_size='3.2B' quantization_level='Q4_K_M'


time: 9.62 s (started: 2025-06-18 13:53:13 -07:00)


## list of models

In [10]:
from ollama import ListResponse, list

response: ListResponse = list()

for model in response.models:
  print('Name:', model.model)
  print('  Size (MB):', f'{(model.size.real / 1024 / 1024):.2f}')
  if model.details:
    print('  Format:', model.details.format)
    print('  Family:', model.details.family)
    print('  Parameter Size:', model.details.parameter_size)
    print('  Quantization Level:', model.details.quantization_level)
  print('\n')

Name: llama3.2:latest
  Size (MB): 1925.84
  Format: gguf
  Family: llama
  Parameter Size: 3.2B
  Quantization Level: Q4_K_M


Name: llama4:16x17b
  Size (MB): 64312.80
  Format: gguf
  Family: llama4
  Parameter Size: 108.6B
  Quantization Level: Q4_K_M


Name: deepseek-r1:latest
  Size (MB): 4983.31
  Format: gguf
  Family: qwen3
  Parameter Size: 8.2B
  Quantization Level: Q4_K_M


Name: gemma3:1b
  Size (MB): 777.55
  Format: gguf
  Family: gemma3
  Parameter Size: 999.89M
  Quantization Level: Q4_K_M


Name: llama3.2-vision:latest
  Size (MB): 7454.48
  Format: gguf
  Family: mllama
  Parameter Size: 10.7B
  Quantization Level: Q4_K_M


Name: llama3.2:1b
  Size (MB): 1259.90
  Format: gguf
  Family: llama
  Parameter Size: 1.2B
  Quantization Level: Q8_0


Name: llama3.3:latest
  Size (MB): 40550.63
  Format: gguf
  Family: llama
  Parameter Size: 70.6B
  Quantization Level: Q4_K_M


time: 9.98 ms (started: 2025-06-18 13:53:23 -07:00)


## Show model status with CPU/GPU usage

In [11]:
from ollama import ProcessResponse, chat, ps, pull

# Ensure at least one model is loaded
response: ListResponse = list()

for model in response.models:
  print(f"Model name: {model.model}")
  print(f"   Size (MB): {(model.size.real / 1024 / 1024):.2f} ; Size: {model.size}" )
  print(f"   Digest: {model.digest}" )
  print(f"   Details: {model.details} ")
  print('\n')

  

Model name: llama3.2:latest
   Size (MB): 1925.84 ; Size: 2019393189
   Digest: a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72
   Details: parent_model='' format='gguf' family='llama' families=['llama'] parameter_size='3.2B' quantization_level='Q4_K_M' 


Model name: llama4:16x17b
   Size (MB): 64312.80 ; Size: 67436862523
   Digest: bf31604e25c25d964e250bcf28a82bfbdbe88af5f236257fabb27629bb24c7f3
   Details: parent_model='' format='gguf' family='llama4' families=['llama4'] parameter_size='108.6B' quantization_level='Q4_K_M' 


Model name: deepseek-r1:latest
   Size (MB): 4983.31 ; Size: 5225376047
   Digest: 6995872bfe4c521a67b32da386cd21d5c6e819b6e0d62f79f64ec83be99f5763
   Details: parent_model='' format='gguf' family='qwen3' families=['qwen3'] parameter_size='8.2B' quantization_level='Q4_K_M' 


Model name: gemma3:1b
   Size (MB): 777.55 ; Size: 815319791
   Digest: 8648f39daa8fbf5b18c7b4e6a8fb4990c692751d49917417b8842ca5758e7ffc
   Details: parent_model='' format

##  Call a function with a model

In [12]:
from ollama import ChatResponse, chat


def add_two_numbers(a: int, b: int) -> int:
  return int(a) + int(b)  # "what is 30 + 12" to produce '3012' instead of 42


def subtract_two_numbers(a: int, b: int) -> int:
  return int(a) - int(b)


# Tools can still be manually defined and passed into chat
subtract_two_numbers_tool = {
  'type': 'function',
  'function': {
    'name': 'subtract_two_numbers',
    'description': 'Subtract two numbers',
    'parameters': {
      'type': 'object',
      'required': ['a', 'b'],
      'properties': {
        'a': {'type': 'integer', 'description': 'The first number'},
        'b': {'type': 'integer', 'description': 'The second number'},
      },
    },
  },
}


messages = [{'role': 'user', 'content': 'What is three plus one?'}]
print('Prompt:', messages[0]['content'])

available_functions = {
  'add_two_numbers': add_two_numbers,
  'subtract_two_numbers': subtract_two_numbers,
}

response: ChatResponse = chat(
  my_model,
  messages=messages,
  tools=[add_two_numbers, subtract_two_numbers_tool],
)

if response.message.tool_calls:
  # There may be multiple tool calls in the response
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    if function_to_call := available_functions.get(tool.function.name):
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)
    else:
      print('Function', tool.function.name, 'not found')

# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'name': tool.function.name})

  # Get final response from model with function outputs
  final_response = chat(my_model, messages=messages)
  print('Final response:', final_response.message.content)

else:
  print('No tool calls returned from model')

Prompt: What is three plus one?
Calling function: add_two_numbers
Arguments: {'a': '3', 'b': '1'}
Function output: 4
Final response: The result of the equation 3 + 1 is indeed 4.
time: 1.33 s (started: 2025-06-18 13:53:23 -07:00)
